# Pangolin RSF package trial

This notebook is a package-integration version of the pangolin RSF workflow.

The goal is not yet to reproduce every exploratory step from `RSF_pangolin.py`, but to test that the new `hsa` package functions work together:

1. load and clean pangolin relocations,
2. define an availability domain,
3. open an environmental raster stack,
4. sample used/available points at multiple resolutions,
5. fit an RSF with `FeatureSpec`,
6. predict an RSF surface,
7. evaluate with a Boyce-style diagnostic,
8. optionally use Dask/HPC helpers.


## 0. Setup

Run this from the repository root after installing the package in editable mode:

```bash
conda env create -f environment.yml
conda activate hsa
pip install -e .
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt

from shapely import Point, Polygon

from hsa import FeatureSpec
from hsa.compute import (
    make_local_dask_client,
    open_raster_stack_zarr,
    persist_if_dask,
    sample_raster_stack_batched,
    suggest_xy_chunks,
)
from hsa.sampling import sample_available_points, sample_raster_stack_multiscale
from hsa.rsf import fit_rsf, predict_rsf_points, predict_rsf_surface_multiscale
from hsa.rsf.validation import boyce_quantile_bins, boyce_sliding_window
from hsa.rsf.cv import cross_validate_rsf_temporal


In [ ]:
# Optional but recommended for local testing.
# On an HPC login node, do not start a large LocalCluster here.
client = make_local_dask_client(n_workers=4, threads_per_worker=1, local_directory='dask-tmp')
client

## 1. Paths and configuration

Adjust these paths to your local repository/data layout. The notebook assumes that the environmental stack has already been written to Zarr by the exploratory Earth Engine workflow.

In [ ]:
DATA = Path('data')
SHAPES = Path('shapefiles')

RELOC_CSV = DATA / 'influx_data_all_2024-07-15-2026-04-02.csv'
PERIMETER_FILE = SHAPES / 'OKJ_Okonjima_Nature_Reserve.shp'
ENV_ZARR = Path('pango_env_32733.zarr')

TARGET_CRS = 'EPSG:32733'
BUFFER_M = 10_000
MIN_POINTS_PER_INDIVIDUAL = 100
AVAILABILITY_FACTOR = 100
TARGET_RESOLUTIONS = [30, 90, 300]


## 2. Load and clean pangolin relocations

This keeps the project-specific cleaning in the notebook. The package should remain general.

In [ ]:
raw = pd.read_csv(RELOC_CSV, low_memory=False)
pangolins = raw.loc[raw['Species'] == 'Pangolin', ['Individual_ID', 'Timestamp', 'Latitude', 'Longitude']].copy()
pangolins['Timestamp'] = pd.to_datetime(pangolins['Timestamp'], errors='coerce')

pangolins = pangolins.dropna(subset=['Timestamp', 'Latitude', 'Longitude'])
pangolins = pangolins.loc[(pangolins['Latitude'] != 0) & (pangolins['Longitude'] != 0)].copy()

reloc = gpd.GeoDataFrame(
    pangolins,
    geometry=gpd.points_from_xy(pangolins['Longitude'], pangolins['Latitude']),
    crs='EPSG:4326',
).to_crs(TARGET_CRS)

reloc.head()

In [ ]:
perimeter_raw = gpd.read_file(PERIMETER_FILE, engine='pyogrio')
perimeter = gpd.GeoDataFrame(geometry=[Polygon(perimeter_raw.geometry.iloc[0])], crs=perimeter_raw.crs).to_crs(TARGET_CRS)
domain = gpd.GeoDataFrame(geometry=perimeter.geometry.buffer(BUFFER_M), crs=TARGET_CRS)

reloc = reloc.loc[reloc.geometry.within(domain.geometry.union_all())].copy()
len(reloc), reloc['Individual_ID'].nunique()

In [ ]:
# Project-specific exclusions from the exploratory workflow.
exclusions = [
    (Point(16.64602802460817, -20.852922284953834), 1000, 'AfriCat / house'),
    (Point(16.637302, -20.861337), 100, 'airfield'),
    (Point(16.800148, -20.846528), 100, 'B1'),
]

for point, radius, label in exclusions:
    buffer_geom = gpd.GeoSeries([point], crs='EPSG:4326').to_crs(TARGET_CRS).buffer(radius).iloc[0]
    before = len(reloc)
    reloc = reloc.loc[~reloc.geometry.within(buffer_geom)].copy()
    print(label, before - len(reloc), 'points removed')

# One point per individual per hour.
reloc['hour'] = reloc['Timestamp'].dt.floor('h')
reloc = reloc.drop_duplicates(subset=['Individual_ID', 'hour']).copy()

counts = reloc.groupby('Individual_ID').size().sort_values()
keep_ids = counts[counts > MIN_POINTS_PER_INDIVIDUAL].index
reloc = reloc.loc[reloc['Individual_ID'].isin(keep_ids)].copy()

counts.loc[keep_ids].describe(), reloc['Individual_ID'].nunique(), len(reloc)

## 3. Open environmental raster stack

This starts from the Zarr stack created in the exploratory workflow. The package can later get reusable Earth Engine stack builders, but for now the project-specific predictor stack should stay here or in `examples/`.

In [ ]:
env = open_raster_stack_zarr(ENV_ZARR, name='env', chunks='auto')
env = env.rio.write_crs(TARGET_CRS) if env.rio.crs is None else env
env = env.chunk(suggest_xy_chunks(env, target_chunk_mb=256))
env = persist_if_dask(env, client=client)

env

In [ ]:
available_bands = [str(b) for b in env.band.values]
available_bands[:20], len(available_bands)

## 4. Sample used and available points

This tests the new `sample_available_points()` and `sample_raster_stack_multiscale()` functions.

In [ ]:
n_available = len(reloc) * AVAILABILITY_FACTOR
samples = sample_available_points(domain, n_available, used=reloc, seed=42, timestamp_col='Timestamp')
samples['used'].value_counts(), samples.crs

In [ ]:
sampled, env_by_scale = sample_raster_stack_multiscale(
    samples,
    env,
    target_resolutions=TARGET_RESOLUTIONS,
    reducer='mean',
)

sampled.shape, list(env_by_scale.keys())

In [ ]:
# Quick sanity check: used/available balance and missingness.
display(sampled['used'].value_counts())
missing = sampled.isna().mean().sort_values(ascending=False)
missing.head(20)

## 5. Choose a first model specification

Start with a deliberately small model. The first notebook goal is to verify package integration, not to find the final ecological model.

In [ ]:
# Candidate continuous predictors. Keep only columns that exist.
candidate_linear = [
    'ndvi_mean_30m',
    'ndvi_mean_90m',
    'vvvh_sd_30m_30m',  # may not exist, depending on source band names
    'dist2water_30m',
]

linear = [c for c in candidate_linear if c in sampled.columns]
linear

In [ ]:
# If the guessed list above is empty, inspect columns and choose manually.
if not linear:
    display([c for c in sampled.columns if 'ndvi' in c.lower()][:30])
    raise ValueError('Choose at least one predictor from sampled.columns and rerun this cell.')

spec = FeatureSpec(linear=linear[:2], add_const=True)
spec

## 6. Fit RSF and predict point-level values

In [ ]:
model, scaler, fitted_spec, meta = fit_rsf(sampled, spec)
print(model.summary())
meta

In [ ]:
pred_points = predict_rsf_points(sampled, model, scaler, fitted_spec, meta)
pred_points[['used', 'rsf_pred']].groupby('used').describe()

## 7. Predict RSF surface

This tests the new multiscale surface code. It will resample non-target scales to the target grid if needed.

In [ ]:
rsf = predict_rsf_surface_multiscale(
    env_by_scale,
    target_scale='30m',
    model=model,
    scaler=scaler,
    spec=fitted_spec,
    meta=meta,
    continuous_resampling='bilinear',
    categorical_resampling='nearest',
)
rsf

In [ ]:
rsf_clip = rsf.rio.clip(perimeter.geometry, perimeter.crs)

fig, ax = plt.subplots(figsize=(8, 7))
rsf_clip.sel(band='rsf').plot(ax=ax, robust=True)
perimeter.boundary.plot(ax=ax, color='black', linewidth=1)
ax.set_aspect('equal')
ax.set_title('Package-trial pangolin RSF')
plt.show()

## 8. Apparent Boyce diagnostic

This is an apparent/in-sample diagnostic. It is useful for package testing, but it should not be presented as final validation.

In [ ]:
B, boyce = boyce_quantile_bins(
    pred_points,
    rsf,
    domain,
    n_background_points=10_000,
    n_bins=20,
    seed=42,
)
B, boyce.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(boyce['rsf_mid'], boyce['pe'], marker='o')
ax.axhline(1, linestyle='--', color='black')
ax.set_xlabel('RSF bin midpoint, log scale')
ax.set_ylabel('Predicted / expected')
ax.set_title(f'Apparent Boyce = {B:.2f}')
plt.show()

## 9. Temporal cross-validation smoke test

This can be slow. Start with small `k_folds`, a small availability factor, and few background points.

In [ ]:
# Optional: uncomment once the basic apparent workflow works.
# cv = cross_validate_rsf_temporal(
#     reloc,
#     env.sel(band=[name.rsplit('_', 1)[0] for name in fitted_spec.linear if name.endswith('30m')]),
#     FeatureSpec(linear=[name.rsplit('_', 1)[0] for name in fitted_spec.linear if name.endswith('30m')]),
#     k_folds=3,
#     sampling_factor_train=10,
#     n_background_boyce=5_000,
# )
# cv

## 10. Remote-sensing diagnostics, optional

Use this when you are back at the Earth Engine predictor-building stage. The reusable functions now live in `hsa.remote_sensing`.

In [ ]:
# Optional sketch only. Requires Earth Engine objects.
# from hsa.remote_sensing import initialize_earth_engine, spatial_summary, temporal_summary
# ee = initialize_earth_engine(project='test-with-greta')
# summary, sample_gdf = spatial_summary(ndvi_mean, aoi_ee, band='ndvi_mean', scale=30, projection=TARGET_CRS, target_crs=TARGET_CRS)
# wide, long = temporal_summary(ndvi_collection, aoi_ee, band='ndvi', scale=30, start='2025-01-01', end='2025-12-31', projection=TARGET_CRS)

## 11. What to check after running

- Does the package import cleanly with `pip install -e .`?
- Does `sample_available_points()` preserve the correct CRS?
- Do `sampled.columns` match the expected multiscale naming convention?
- Does `FeatureSpec` fit a simple model without singularity warnings?
- Does `predict_rsf_surface_multiscale()` produce a raster with the expected CRS and extent?
- Does Boyce behave sensibly when changing predictors?
- Which pieces still feel pangolin-specific and belong in this notebook rather than in `src/hsa`?
